# Trustworthy Anomaly Agent on ESA-ADB — end-to-end walkthrough

This notebook runs the **complete pipeline** of the project on real satellite telemetry
(ESA's Mission2, lightweight subset): it detects anomalies, calibrates how much you can
trust each detection, explains which channels caused it, and generates an operator alert
whose text is audited by an LLM judge before it reaches you. Along the way it explains
the **key design decisions** of each module: not just what runs, but *why it is built
this way*. For the project context (the operator's problem, the dataset, the
trustworthiness gap) see the [README](../README.md).

```
Telemetry (Mission2, channels 18-28)
  → [1] Detection      — Windowed Isolation Forest        → anomaly score per window
  → [2] Uncertainty    — conformal prediction             → calibrated confidence
  → [3] Attribution    — channel-level explanation        → which sensors caused it
  → [4] Alert layer    — grounded LLM brief + judge       → auditable operator alert
  → [5] This notebook  — the whole chain, live
```

**How to use this notebook.** Reading it on GitHub (outputs are committed) takes
~15-20 min and requires nothing. Running it yourself requires downloading the dataset
once (~3.8 GB, instructions below); after the one-time preprocessing (~30 min) every
later run skips the heavy steps automatically.

---
**Data attribution.** ESA Anomaly Dataset (ESA-ADB) — © European Space Agency (ESOC),
KP Labs, Airbus Defence and Space — licensed [CC BY 3.0 IGO](https://creativecommons.org/licenses/by/3.0/igo/).
Dataset: [10.5281/zenodo.12528696](https://doi.org/10.5281/zenodo.12528696) ·
Paper: [arXiv:2406.17826](https://arxiv.org/abs/2406.17826) ·
Code: [kplabs-pl/ESA-ADB](https://github.com/kplabs-pl/ESA-ADB) (MIT), minimally
vendored under `src/` (see `NOTICE`). All results below are traceable to the dataset —
no invented data.


## 0 · Setup & data

The pipeline needs the raw Mission2 telemetry **once**. If you only want to read this
notebook, skip ahead — every cell below is already executed.

1. Download the **ESA-Mission2** folder (~3.8 GB) from Zenodo:
   [doi.org/10.5281/zenodo.12528696](https://doi.org/10.5281/zenodo.12528696)
2. Place it at `data/ESA-Mission2/` (so that `data/ESA-Mission2/labels.csv` exists).
3. Run the cells below. The first run preprocesses the raw channels into the
   train/test CSVs (~30 min, one-time); later runs detect the existing output and skip.

Everything is deterministic: same input, same bytes out — down to the final
F0.5 = 0.9487. There is no hidden state and no cached shortcut in the pipeline itself.


In [1]:
import sys, time, subprocess
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))  # notebook lives in notebooks/
import config

TIMINGS: dict = {}  # wall-clock per heavy step, summarized at the end

RAW_DIR = config.REPO / "data" / "ESA-Mission2"
preprocessed = config.TEST_CSV.exists() and config.TRAIN_CSV.exists()

if preprocessed:
    print(f"[ok] Preprocessed CSVs found — ready.\n     {config.TEST_CSV.name}, {config.TRAIN_CSV.name}")
elif RAW_DIR.exists():
    print("[ok] Raw dataset found — the next cell will preprocess it (~30 min, one-time).")
else:
    raise SystemExit(
        "Dataset not found.\n"
        f"  Expected raw data at:        {RAW_DIR}\n"
        f"  (or preprocessed CSVs at:    {config.PREP_DIR})\n"
        "  Download ESA-Mission2 (~3.8 GB): https://doi.org/10.5281/zenodo.12528696\n"
        "  Then re-run this cell. See section '0 · Setup & data' above."
    )


[ok] Preprocessed CSVs found — ready.
     21_months.test.csv, 21_months.train.csv


In [2]:
# [1a] Raw telemetry -> canonical train/test CSVs (ESA's own prep, vendored — see NOTICE).
# Deterministic: depends ONLY on the immutable Zenodo raw data + preprocessing.py.
# To force regeneration (e.g. after editing the script): delete data/preprocessed_subset/.
if preprocessed:
    print("[skip] Already preprocessed — ~30 min saved.")
else:
    print("[run ] Preprocessing channels 18-28 (~30 min on Apple Silicon)...")
    t0 = time.perf_counter()
    subprocess.run(
        [sys.executable, str(config.REPO / "src" / "m1_detection" / "preprocessing.py"), str(RAW_DIR)],
        check=True,
    )
    TIMINGS["preprocess"] = time.perf_counter() - t0
    print(f"[done] Preprocessing finished in {TIMINGS['preprocess']/60:.1f} min.")


[skip] Already preprocessed — ~30 min saved.


## 1 · Detection — Windowed Isolation Forest

**What runs here.** ESA's own winning detector for this benchmark subset
(`subsequence_if`), executed **verbatim** from the vendored copy in
`src/m1_detection/vendor/algorithm.py`. We *reproduce* the paper's headline number —
reproduce meaning: re-run ESA's published experiment with their own code, data and
metric, and land on their number (they report **F0.5 = 0.949**; we get **0.9487**).
That number then doubles as a regression alarm: if any later change moves it,
something broke.

**Why a simple forest and not deep learning?** The ESA-ADB paper's own finding: on the
*full* benchmark, every tested detector collapses (F0.5 ≈ 0.07) — nobody, not even the
authors, can tell in advance which detector to trust on which data (*No Free Lunch*).
That is exactly why this project's contribution is the **trust layer** (modules 2-4)
rather than yet another detector: an operator facing a bare 0/1 flag has no way to know
when to believe it.

**How it works, in three lines.** A sliding window of 17 samples (~5 min of telemetry
at 18 s) slides over the 11 channels; each window is flattened into one 187-dimensional
point; an **Isolation Forest** (Liu 2008) tries to isolate each point with random cuts —
the fewer cuts needed, the more anomalous the window. This is the core of ESA's
`algorithm.py` (quoted, not re-implemented):

```python
# src/m1_detection/vendor/algorithm.py (condensed) — the entire detection logic
data = sliding_window_view(data, window_shape=17, axis=0).reshape(-1, 17 * n_channels)
clf = IForest(n_estimators=200, random_state=42, ...)
clf.fit(data)                       # train: learn what "normal" looks like
result = clf.predict(data)          # execute: 0/1 per window
result = np.pad(result, 17 // 2)    # window center -> per-point flag
```

**Design decisions worth knowing:**

1. **Vendor, never re-implement.** The detector, the preprocessing and the metric are
   ESA's own files, byte-identical (see `NOTICE`). Reproducing 0.949 with their code is
   *evidence*; re-implementing it would only prove we can introduce bugs.
2. **`subsequence_if` is the *standard* Isolation Forest** — not the Extended variant.
   ESA also ships `subsequence_eif` (one letter away, a different algorithm, not used
   in this benchmark). Documented because the confusion is expensive.
3. **Verify the data before trusting any result.** The preprocessing output was checked
   against ESA's shipped metadata (grid lengths, anomaly lengths, split boundaries — 6
   external checks, see [docs/REPRODUCTION.md](../docs/REPRODUCTION.md)) *before*
   running any detector.
4. **Detector-agnostic by construction.** Modules 2-4 consume only the detector's
   *outputs* — a continuous score per window, the windowed data, a scoring function —
   never its internals. Swapping in a different detector (e.g. ESA's deep DC-VAE)
   means regenerating those artifacts, not rewriting the trust layer.

**One limitation drives the next section:** `predict()` emits a hard 0/1. It says
*"anomaly"* with the same face whether it is 99% sure or 51% sure. For an operator that
difference is everything — quantifying it is Module 2's job.


In [ ]:
from m1_detection import model as m1_model

MODEL_PKL = config.CACHE_DIR / "model.pkl"
SCORES_TEST = config.CACHE_DIR / "scores_test.csv"

if MODEL_PKL.exists() and SCORES_TEST.exists():
    print("[skip] Trained model + test scores found — ~11 min saved.")
    print("       Deterministic (seed=42): retraining reproduces them byte-identically.")
    print("       To force it: delete both files from data/cached/ and re-run.")
else:
    print("[run ] Training Windowed iForest (21 months) + scoring the test set (~11 min)...")
    t0 = time.perf_counter()
    m1_model.main()
    TIMINGS["train+score"] = time.perf_counter() - t0
    print(f"[done] {TIMINGS['train+score']/60:.1f} min.")


In [ ]:
import io, contextlib
from m1_detection import evaluation as m1_eval

with contextlib.redirect_stdout(io.StringIO()):   # ESA's metric prints verbose internals
    result = m1_eval.evaluate()
f05 = result["EW_F_0.50"]
print(f"EW_F_0.50    = {f05:.4f}   (paper Table 2: 0.949)")
print(f"EW_precision = {result['EW_precision']:.4f}")
print(f"EW_recall    = {result['EW_recall']:.4f}")
assert abs(f05 - 0.9487) < 1e-3, "Regression: the pipeline no longer reproduces the paper's number"


In [ ]:
# The detection on real telemetry: longest predicted anomalous stretch, chosen
# programmatically from the detector's own output (no hand-picked indices).
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

flags = np.loadtxt(SCORES_TEST, delimiter=",")            # detector 0/1 per test point
edges = np.flatnonzero(np.diff(np.r_[0, flags, 0]))       # run boundaries
starts, ends = edges[::2], edges[1::2]
k = int(np.argmax(ends - starts))                         # longest predicted event
pad = ends[k] - starts[k]
lo, hi = max(0, starts[k] - pad), min(len(flags), ends[k] + pad)

seg = pd.read_csv(config.TEST_CSV, skiprows=range(1, lo + 1), nrows=hi - lo)
seg["timestamp"] = pd.to_datetime(seg["timestamp"])
pred = flags[lo:hi].astype(bool)

# plot the two channels with the most annotated anomaly in this stretch
anno_cols = [f"is_anomaly_{c}" for c in config.TARGET_CHANNELS]
top2 = (seg[anno_cols] > 0).sum().nlargest(2).index
fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
for ax, col in zip(axes, top2):
    ch = col.replace("is_anomaly_", "")
    truth = (seg[col] > 0).to_numpy()
    ax.plot(seg["timestamp"], seg[ch], lw=0.7, color="tab:blue")
    ax.fill_between(seg["timestamp"], *ax.get_ylim(), where=truth,
                    color="gray", alpha=0.3, label="ESA annotation")
    ax.fill_between(seg["timestamp"], *ax.get_ylim(), where=pred,
                    color="tab:red", alpha=0.15, label="detector flag")
    ax.set_ylabel(ch)
axes[0].legend(loc="upper right", fontsize=8)
axes[0].set_title("Longest detected event — telemetry, ESA ground truth (gray), detector flag (red)")
plt.tight_layout()
plt.show()


## 2 · Uncertainty — conformal prediction

*(section coming in Fase 3)*

## 3 · Attribution — which channels caused it

*(section coming in Fase 4)*

## 4 · Trustworthy alert — grounded brief + LLM judge

*(section coming in Fase 6)*

## 5 · Live streaming simulation

*(section coming in Fase 7)*

## 6 · Wrap-up — what this is *not*, future work, references

*(section coming in Fase 8)*